#### Reconnect to the database
#### New notebook, so we rebuild the SQLAlchemy engine fresh here.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, URL, text

conn_url = URL.create(
    "mssql+pyodbc", host=r"localhost\SQLEXPRESS",
    database="BA_Airline_Commercial_Analytics",
    query={"driver": "ODBC Driver 17 for SQL Server", "trusted_connection": "yes"},
)
engine = create_engine(conn_url)

print("connected")

connected


#### Reconnect to the database and cache the data we need locally
#### New notebook, so the engine is rebuilt fresh. Pulling both views once and
#### saving them as parquet files means the rest of this notebook's forecasting
#### and modelling work never has to re-query SQL Server.

In [2]:
import pandas as pd
from sqlalchemy import create_engine, URL
import os

conn_url = URL.create(
    "mssql+pyodbc", host=r"localhost\SQLEXPRESS",
    database="BA_Airline_Commercial_Analytics",
    query={"driver": "ODBC Driver 17 for SQL Server", "trusted_connection": "yes"},
)
engine = create_engine(conn_url)

os.makedirs(r"..\data\processed", exist_ok=True)

gold_df = pd.read_sql("SELECT * FROM gold.vw_RouteCompetitivePerformance", engine)
gold_df.to_parquet(r"..\data\processed\route_competitive_performance.parquet")

trends_df = pd.read_sql("SELECT * FROM silver.vw_RouteFareTrends", engine)
trends_df.to_parquet(r"..\data\processed\route_fare_trends.parquet")

print("gold rows:", len(gold_df))
print("trends rows:", len(trends_df))
print("both saved to data/processed/")

C:\Users\HARDEEP\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


gold rows: 1274
trends rows: 71937
both saved to data/processed/


#### Check which routes have enough history to forecast reliably
#### Counting years of data per route in the trends dataset, so we pick routes
#### with real depth rather than ones with just a handful of scattered years.

In [3]:
trends_df = pd.read_parquet(r"..\data\processed\route_fare_trends.parquet")

route_history = trends_df.groupby(['airport_1', 'airport_2'])['Year'].agg(
    years_of_data='count',
    earliest='min',
    latest='max'
).reset_index()

route_history_sorted = route_history.sort_values('years_of_data', ascending=False)
print(route_history_sorted.head(15))
print("\nroutes with 20+ years of data:", (route_history['years_of_data'] >= 20).sum())
print("total distinct routes in trends data:", len(route_history))

     airport_1 airport_2  years_of_data  earliest  latest
1033       CLT       SJC             31      1993    2024
1309       DEN       HOU             31      1993    2024
1307       DEN       EWR             31      1993    2024
2813       MDW       OKC             31      1993    2024
2814       MDW       OMA             31      1993    2024
2815       MDW       ONT             31      1993    2024
2816       MDW       ORF             31      1993    2024
1302       DEN       DCA             31      1993    2024
1301       DEN       BWI             31      1993    2024
1300       DEN       BUR             31      1993    2024
2817       MDW       PBI             31      1993    2024
2818       MDW       PDX             31      1993    2024
1297       DCA       PBI             31      1993    2024
2820       MDW       PHL             31      1993    2024
2821       MDW       PHX             31      1993    2024

routes with 20+ years of data: 2034
total distinct routes in trends dat

#### Forecast average fare trend for DEN -> HOU
#### Using the full 31-year history for this route. Since this is yearly data
#### (not monthly), we use a simple trend model rather than a seasonal one -
#### seasonality needs repeating patterns within a year, and we only have one
#### data point per year here.

In [4]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

route_ts = trends_df[
    (trends_df['airport_1'] == 'DEN') & (trends_df['airport_2'] == 'HOU')
].sort_values('Year')

ts = route_ts.set_index('Year')['avg_fare_lg']

train = ts.iloc[:-5]
test = ts.iloc[-5:]

model = ExponentialSmoothing(train, trend='add', seasonal=None).fit()
forecast = model.forecast(5)

mape = (abs((test - forecast) / test)).mean() * 100

print("Actual last 5 years:")
print(test)
print("\nForecasted for those same 5 years:")
print(forecast)
print(f"\nMAPE: {mape:.2f}%")

Actual last 5 years:
Year
2020    146.0625
2021    136.0700
2022    178.0950
2023    194.2725
2024    186.7600
Name: avg_fare_lg, dtype: float64

Forecasted for those same 5 years:
26    171.818563
27    171.384507
28    170.950450
29    170.516393
30    170.082337
dtype: float64

MAPE: nan%


C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


#### Fix the index issue and re-test on pre-COVID years
#### The forecast/actual comparison failed because Year wasn't a real time
#### index. Fixing that. Also testing on 2015-2019 (pre-COVID) separately from
#### 2020-2024, since COVID represents a structural break no model trained on
#### earlier data could reasonably predict.

In [6]:
route_ts = trends_df[
    (trends_df['airport_1'] == 'DEN') & (trends_df['airport_2'] == 'HOU')
].sort_values('Year')

ts = route_ts.set_index('Year')['avg_fare_lg']

# Pre-COVID test: train on everything through 2014, test on 2015-2019
train_normal = ts[ts.index <= 2014]
test_normal = ts[(ts.index >= 2015) & (ts.index <= 2019)]

model_normal = ExponentialSmoothing(train_normal, trend='add', seasonal=None).fit()
forecast_normal = model_normal.forecast(len(test_normal))
forecast_normal.index = test_normal.index  # <-- the actual fix: assign real years directly

mape_normal = (abs((test_normal - forecast_normal) / test_normal)).mean() * 100

print("=== Pre-COVID test (2015-2019) ===")
print("Actual:", test_normal.values.round(2))
print("Forecast:", forecast_normal.values.round(2))
print(f"MAPE: {mape_normal:.2f}%")

# Full history including COVID, for comparison
train_full = ts[ts.index <= 2019]
test_full = ts[ts.index >= 2020]

model_full = ExponentialSmoothing(train_full, trend='add', seasonal=None).fit()
forecast_full = model_full.forecast(len(test_full))
forecast_full.index = test_full.index

mape_full = (abs((test_full - forecast_full) / test_full)).mean() * 100
print(f"\nMAPE including COVID disruption (2020-2024): {mape_full:.2f}%")

=== Pre-COVID test (2015-2019) ===
Actual: [198.77 188.5  178.88 182.35 172.25]
Forecast: [196.73 197.37 198.02 198.66 199.31]
MAPE: 8.22%

MAPE including COVID disruption (2020-2024): 13.75%


C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.p

#### Pricing elasticity: does a higher fare relate to fewer passengers?
#### Using the gold view's 1,274 routes (2024). Checking whether fare and
#### passenger volume move in opposite directions (classic price sensitivity)
#### or whether market dominance breaks that relationship.

In [7]:
gold_df = pd.read_parquet(r"..\data\processed\route_competitive_performance.parquet")

correlation = gold_df[['avg_fare_lg', 'total_passengers', 'avg_large_ms']].corr()
print(correlation)

print("\nRoutes split by market dominance:")
high_dominance = gold_df[gold_df['avg_large_ms'] >= 0.7]
low_dominance = gold_df[gold_df['avg_large_ms'] < 0.7]

print(f"\nHigh dominance (70%+ market share), {len(high_dominance)} routes:")
print("avg fare:", high_dominance['avg_fare_lg'].mean().round(2))
print("avg passengers:", high_dominance['total_passengers'].mean().round(0))

print(f"\nLow dominance (<70% market share), {len(low_dominance)} routes:")
print("avg fare:", low_dominance['avg_fare_lg'].mean().round(2))
print("avg passengers:", low_dominance['total_passengers'].mean().round(0))

                  avg_fare_lg  total_passengers  avg_large_ms
avg_fare_lg          1.000000          0.117280     -0.184861
total_passengers     0.117280          1.000000     -0.394219
avg_large_ms        -0.184861         -0.394219      1.000000

Routes split by market dominance:

High dominance (70%+ market share), 679 routes:
avg fare: 230.04
avg passengers: 350.0

Low dominance (<70% market share), 595 routes:
avg fare: 251.47
avg passengers: 809.0


#### Build a reusable route analysis tool + demonstrate market dominance
#### Wraps the forecast logic into a function that works for any route. Also
#### surfaces the routes with the strongest single-carrier dominance in the
#### dataset -- the same large_ms metric would apply directly to a market like
#### IndiGo in India if we had that data.

In [8]:
def analyze_route(origin, dest, trends_df, split_year=2019):
    route_ts = trends_df[
        (trends_df['airport_1'] == origin) & (trends_df['airport_2'] == dest)
    ].sort_values('Year')

    if len(route_ts) < 8:
        return f"Not enough history for {origin}->{dest} ({len(route_ts)} years) -- need at least 8 for a reliable forecast."

    ts = route_ts.set_index('Year')['avg_fare_lg']
    train = ts[ts.index <= split_year]
    test = ts[ts.index > split_year]

    if len(train) < 5 or len(test) == 0:
        return f"Not enough data on either side of {split_year} for {origin}->{dest}."

    model = ExponentialSmoothing(train, trend='add', seasonal=None).fit()
    forecast = model.forecast(len(test))
    forecast.index = test.index

    mape = (abs((test - forecast) / test)).mean() * 100

    print(f"=== {origin} -> {dest} ===")
    print(f"History: {ts.index.min()}-{ts.index.max()} ({len(ts)} years)")
    print(f"Test years ({split_year+1}-{ts.index.max()}):")
    print("  Actual:", test.values.round(2))
    print("  Forecast:", forecast.values.round(2))
    print(f"  MAPE: {mape:.2f}%")
    return mape

# Test it on a route we haven't looked at yet
analyze_route('ATL', 'LAX', trends_df)

print("\n" + "="*50)

# The market dominance question - which routes are most like IndiGo-style monopolies?
most_dominated = gold_df.sort_values('avg_large_ms', ascending=False)[
    ['origin', 'dest', 'avg_large_ms', 'avg_fare_lg', 'total_passengers']
].head(10)
print("\nRoutes with strongest single-carrier dominance (2024):")
print(most_dominated)

=== ATL -> LAX ===
History: 1993-2024 (31 years)
Test years (2020-2024):
  Actual: [311.87 295.24 446.58 460.06 461.1 ]
  Forecast: [349.47 350.69 351.9  353.12 354.33]
  MAPE: 19.69%


Routes with strongest single-carrier dominance (2024):
    origin dest  avg_large_ms  avg_fare_lg  total_passengers
0      ATW  AZA           1.0       168.30               206
900    SWF  PIE           1.0       119.44                81
893    IND  PIE           1.0       104.50               156
894    LCK  PIE           1.0       108.74                80
895    MCI  PIE           1.0        94.75                34
896    OMA  PIE           1.0       106.95                80
897    PIT  PIE           1.0        93.43                60
898    RIC  PIE           1.0        81.29                20
899    SDF  PIE           1.0        85.16                30
901    SYR  PIE           1.0       136.11               122


C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\HARDEEP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


#### Pricing elasticity: does a higher fare relate to fewer passengers?
#### Using the gold view's 1,274 routes. Splitting by market dominance to see
#### whether the fare-vs-passengers relationship changes for near-monopoly
#### routes versus contested ones -- same underlying logic as the IndiGo
#### question, applied to what we actually have data for.

In [9]:
correlation = gold_df[['avg_fare_lg', 'total_passengers', 'avg_large_ms']].corr()
print("Correlation matrix:")
print(correlation)

high_dominance = gold_df[gold_df['avg_large_ms'] >= 0.7]
low_dominance = gold_df[gold_df['avg_large_ms'] < 0.7]

print(f"\nHigh dominance (70%+ market share), {len(high_dominance)} routes:")
print("avg fare:", high_dominance['avg_fare_lg'].mean().round(2))
print("avg passengers:", high_dominance['total_passengers'].mean().round(0))

print(f"\nLow dominance (<70% market share), {len(low_dominance)} routes:")
print("avg fare:", low_dominance['avg_fare_lg'].mean().round(2))
print("avg passengers:", low_dominance['total_passengers'].mean().round(0))

Correlation matrix:
                  avg_fare_lg  total_passengers  avg_large_ms
avg_fare_lg          1.000000          0.117280     -0.184861
total_passengers     0.117280          1.000000     -0.394219
avg_large_ms        -0.184861         -0.394219      1.000000

High dominance (70%+ market share), 679 routes:
avg fare: 230.04
avg passengers: 350.0

Low dominance (<70% market share), 595 routes:
avg fare: 251.47
avg passengers: 809.0
